In [19]:
import os
import glob
import random
import json
import numpy as np
import cv2 as cv
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import torchvision.transforms as transforms
from tqdm import tqdm
import matplotlib.pyplot as plt

# --- CONFIGURACIÓN ---
DATASET_DIR = "./dataset/training_collages"
MODEL_SAVE_PATH = "unet_multiclass.pth"
# Esta imagen de validación vendrá del script render_in_blender.py
IMG_VALIDATION_PATH = "./dataset/virtual_sample_blender/render_final_top.png"

# Parámetros
BATCH_SIZE = 16
EPOCHS = 10        
LR = 1e-3
IMG_SIZE = 256     
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# --- ARQUITECTURA U-NET (3 CLASES) ---
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=3): 
        super().__init__()
        self.inc = DoubleConv(n_channels, 32)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(32, 64))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv1 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = DoubleConv(128, 64)
        self.up3 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv3 = DoubleConv(64, 32)
        self.outc = nn.Conv2d(32, n_classes, 1)
    def forward(self, x):
        x1 = self.inc(x); x2 = self.down1(x1); x3 = self.down2(x2); x4 = self.down3(x3)
        x = self.up1(x4); x = torch.cat([x, x3], dim=1); x = self.conv1(x)
        x = self.up2(x); x = torch.cat([x, x2], dim=1); x = self.conv2(x)
        x = self.up3(x); x = torch.cat([x, x1], dim=1); x = self.conv3(x)
        return self.outc(x)

# --- Celda 3: Dataset Loader y Entrenamiento (CORREGIDO) ---
class ThreeClassDataset(Dataset):
    def __init__(self, root_dir, crop_size=512): # Subimos a 512 para más contexto
        self.images = sorted(glob.glob(os.path.join(root_dir, "images", "*.png")))
        self.masks_dir = os.path.join(root_dir, "masks")
        self.instances_dir = os.path.join(root_dir, "instances")
        self.crop_size = crop_size
        
    def __len__(self): return len(self.images)

    def load_set(self, idx):
        img_path = self.images[idx]
        base_name = os.path.basename(img_path)
        # Ajuste de extensión por si las máscaras son jpg o png
        mask_name = base_name.replace('.jpg', '.png') 
        
        mask_path = os.path.join(self.masks_dir, mask_name)
        inst_path = os.path.join(self.instances_dir, mask_name)
        
        # Cargar imágenes en resolución ORIGINAL (2048x2048)
        img = cv.imread(img_path, cv.IMREAD_UNCHANGED) 
        mask = cv.imread(mask_path, cv.IMREAD_GRAYSCALE)
        inst = cv.imread(inst_path, cv.IMREAD_GRAYSCALE)
        
        return img, mask, inst

    def get_borders(self, inst_mask):
        """Kernel más grueso para forzar separación"""
        kernel = np.ones((5,5), np.uint8) 
        gradient = cv.morphologyEx(inst_mask, cv.MORPH_GRADIENT, kernel)
        borders = np.zeros_like(inst_mask, dtype=np.uint8)
        borders[(gradient > 0)] = 255
        return borders

    def __getitem__(self, idx):
        img, mask, inst = self.load_set(idx)
        
        if img is None: return torch.zeros((3, self.crop_size, self.crop_size)), torch.zeros((self.crop_size, self.crop_size), dtype=torch.long)

        # --- CORRECCIÓN CLAVE: RANDOM CROP ---
        # En lugar de resize, cortamos un pedazo aleatorio
        h, w = mask.shape
        # Si la imagen es más pequeña que el crop, hacemos resize (fallback)
        if h < self.crop_size or w < self.crop_size:
             img = cv.resize(img, (self.crop_size, self.crop_size))
             mask = cv.resize(mask, (self.crop_size, self.crop_size), interpolation=cv.INTER_NEAREST)
             inst = cv.resize(inst, (self.crop_size, self.crop_size), interpolation=cv.INTER_NEAREST)
        else:
            # Coordenadas aleatorias
            top = random.randint(0, h - self.crop_size)
            left = random.randint(0, w - self.crop_size)
            
            img = img[top:top+self.crop_size, left:left+self.crop_size]
            mask = mask[top:top+self.crop_size, left:left+self.crop_size]
            inst = inst[top:top+self.crop_size, left:left+self.crop_size]

        # Calcular bordes SOBRE EL RECORTE (ahorra cómputo)
        border = self.get_borders(inst)
        
        # Armar mapa de clases: 0=Fondo, 1=Arena, 2=Borde
        final_mask = np.zeros_like(mask, dtype=np.int64)
        final_mask[mask > 0] = 1 
        final_mask[border > 0] = 2 

        # Normalización
        if len(img.shape) == 3 and img.shape[2] == 4: # Si tiene alpha
            img = img[:,:,:3] # Descartar alpha por ahora para simplificar
            
        img_tensor = torch.from_numpy(img).permute(2,0,1).float() / 255.0
        mask_tensor = torch.from_numpy(final_mask).long()

        return img_tensor, mask_tensor


In [22]:

# ENTRENAMIENTO
def train():
    ds = ThreeClassDataset(DATASET_DIR)
    if len(ds) == 0:
        print(f"❌ ERROR: No se encontraron imágenes en {DATASET_DIR}")
        return

    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
    model = UNet(n_channels=3, n_classes=3).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    
    # Pesos: Fondo=1, Interior=1, Borde=10 (Dar más importancia al borde)
    weights = torch.tensor([1.0, 1.0, 10.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    print("🚀 Entrenando U-Net 3 Clases (Bordes por Instancia)...")
    
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        progress = tqdm(loader, desc=f"Epoca {epoch+1}")
        
        for img, msk in progress:
            img, msk = img.to(DEVICE), msk.to(DEVICE)
            optimizer.zero_grad()
            pred = model(img)
            loss = criterion(pred, msk)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            progress.set_postfix({"loss": loss.item()})
            
    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print("✅ Modelo guardado.")

if __name__ == "__main__":
    train()

🚀 Entrenando U-Net 3 Clases (Bordes por Instancia)...


Epoca 10: 100%|██████████| 125/125 [1:21:13<00:00, 38.99s/it, loss=0.00478]

✅ Modelo guardado.
